In [1]:

import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
from PIL import Image, ImageDraw, ImageFont
import os

In [2]:
# --- PATHS ---
MODEL_PATH = r"C:\Users\HP\Desktop\SignSight\model_training_updated\Final_15_signs_high_accuracy.h5"  # un latest model name podu
FONT_PATH = r"C:\Windows\Fonts\Nirmala.ttf"  # Tamil font

In [3]:
# Load Model
print("Loading Model...")
model = load_model(MODEL_PATH)

Loading Model...


In [4]:
# Load Font
try:
    font = ImageFont.truetype(FONT_PATH, 60)  # bigger font
except:
    print("Warning: Nirmala Font not found. Using default.")
    font = ImageFont.load_default()

In [5]:
# Actions - YOUR 15 SIGNS
actions = np.array([
    'Beautiful', 'Drink', 'Eat', 'Five', 'Good', 'Hello', 'House', 'Love',
    'Man', 'Mother', 'Run', 'Thank you', 'White', 'Yellow', 'You'
])

# Translation (English + Tamil only)
translate = {
    'Beautiful': {'en': 'Beautiful', 'ta': 'அழகு'},
    'Drink': {'en': 'Drink', 'ta': 'குடி'},
    'Eat': {'en': 'Eat', 'ta': 'சாப்பிடு'},
    'Five': {'en': 'Five', 'ta': 'ஐந்து'},
    'Good': {'en': 'Good', 'ta': 'நல்ல'},
    'Hello': {'en': 'Hello', 'ta': 'வணக்கம்'},
    'House': {'en': 'House', 'ta': 'வீடு'},
    'Love': {'en': 'Love', 'ta': 'காதல்'},
    'Man': {'en': 'Man', 'ta': 'ஆண்'},
    'Mother': {'en': 'Mother', 'ta': 'அம்மா'},
    'Run': {'en': 'Run', 'ta': 'ஓடு'},
    'Thank you': {'en': 'Thank you', 'ta': 'நன்றி'},
    'White': {'en': 'White', 'ta': 'வெள்ளை'},
    'Yellow': {'en': 'Yellow', 'ta': 'மஞ்சள்'},
    'You': {'en': 'You', 'ta': 'நீ'}
}

In [11]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

In [13]:
# Helper Function: Unicode Text Drawing
def draw_unicode_text(img, text, position, color):
    pil_img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(pil_img)
    draw.text(position, text, font=font, fill=color)
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

# Keypoint Extraction (1662 features - new holistic)
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])

In [14]:
# MAIN LOOP
sequence = []
cap = cv2.VideoCapture(0)

# Camera settings
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

cv2.namedWindow('SIGN SIGHT - Sri Lanka Sign Language AI', cv2.WINDOW_NORMAL)
cv2.resizeWindow('SIGN SIGHT - Sri Lanka Sign Language AI', 1000, 600)

with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    print("AI STARTING... DO A SIGN DA MCHA!")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = holistic.process(image_rgb)
        image = frame.copy()

        # Draw landmarks
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

        # Collect keypoints
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-45:]  # 45 frames

        # Status bar
        cv2.rectangle(image, (0,0), (1280, 80), (0,0,0), -1)
        image = draw_unicode_text(image, "SIGN SIGHT - Sri Lanka Sign Language Recognition", (20, 15), (255, 255, 255))

        # Prediction
        if len(sequence) == 45:
            prediction = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]
            action = actions[np.argmax(prediction)]
            confidence = np.max(prediction)

            # Confidence text
            conf_text = f"Detected: {action} ({int(confidence*100)}%)"
            image = draw_unicode_text(image, conf_text, (20, 100), (0, 255, 0) if confidence > 0.6 else (0, 0, 255))

            if confidence > 0.6:
                en_text = f"English: {translate[action]['en']}"
                ta_text = f"தமிழ்: {translate[action]['ta']}"
                image = draw_unicode_text(image, en_text, (20, 180), (0, 255, 0))
                image = draw_unicode_text(image, ta_text, (20, 260), (0, 255, 255))

        cv2.imshow('SIGN SIGHT - Sri Lanka Sign Language AI', image)

        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("LIVE DEMO CLOSED — YOU ARE THE KING DA MCHA!")

AI STARTING... DO A SIGN DA MCHA!


ValueError: in user code:

    File "D:\anaconda3\envs\mp_env\lib\site-packages\keras\src\engine\training.py", line 2341, in predict_function  *
        return step_function(self, iterator)
    File "D:\anaconda3\envs\mp_env\lib\site-packages\keras\src\engine\training.py", line 2327, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "D:\anaconda3\envs\mp_env\lib\site-packages\keras\src\engine\training.py", line 2315, in run_step  **
        outputs = model.predict_step(data)
    File "D:\anaconda3\envs\mp_env\lib\site-packages\keras\src\engine\training.py", line 2283, in predict_step
        return self(x, training=False)
    File "D:\anaconda3\envs\mp_env\lib\site-packages\keras\src\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "D:\anaconda3\envs\mp_env\lib\site-packages\keras\src\engine\input_spec.py", line 298, in assert_input_compatibility
        raise ValueErr

Loading Model...
AI STARTING... DO AN ACTION!


In [4]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model

# ------------------- LOAD MODEL -------------------
print("Loading model...")
model = load_model('Final_15_signs_high_accuracy.h5')
print("Model loaded successfully!")

# ------------------- ACTIONS -------------------
actions = np.array([
    'Beautiful', 'Drink', 'Eat', 'Five', 'Good', 'Hello', 'House', 'Love',
    'Man', 'Mother', 'Run', 'Thank you', 'White', 'Yellow', 'You'
])

sequence_length = 45

# ------------------- MEDIAPIPE SETUP -------------------
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils  # optional - to draw landmarks

def extract_keypoints(results):
    pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[lm.x, lm.y, lm.z] for lm in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

# ------------------- MAIN LOOP -------------------
sequence = []
prediction_text = "Initializing..."

cap = cv2.VideoCapture(0)  # 0 = default camera

# Check if camera opened
if not cap.isOpened():
    print("ERROR: Cannot open camera! Check:")
    print("1. Camera connected?")
    print("2. Another app using camera? (Zoom, Teams, etc.)")
    print("3. Camera permission allowed?")
    exit()

print("Camera opened successfully! Press 'q' to quit.")

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame")
            break

        # Mirror view
        frame = cv2.flip(frame, 1)

        # Mediapipe processing
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = holistic.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Optional: Draw landmarks (helps debug)
        mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION)
        mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS)
        mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
        mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

        # Extract keypoints
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)

        # Keep only last 45 frames
        if len(sequence) > sequence_length:
            sequence = sequence[-sequence_length:]

        # Predict when full sequence
        if len(sequence) == sequence_length:
            input_data = np.expand_dims(sequence, axis=0)
            pred = model.predict(input_data, verbose=0)[0]
            confidence = np.max(pred)
            pred_index = np.argmax(pred)

            if confidence > 0.7:
                prediction_text = f"{actions[pred_index]} ({confidence:.2f})"
            else:
                prediction_text = f"Detecting... ({confidence:.2f})"
        else:
            prediction_text = f"Collecting frames... {len(sequence)}/45"

        # Display prediction
        cv2.putText(image, prediction_text, (10, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)

        # Show frame
        cv2.imshow('SignSight - Real Time Prediction', image)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("Camera closed.")

Loading model...
Model loaded successfully!
Camera opened successfully! Press 'q' to quit.
Camera closed.
